# YOLO-World COCO Producer (Colab)
Bu defter 1.9M+ görüntüyü YOLO-World ile tarayıp COCO JSON üretecek şekilde tasarlandı. Varsayılan çıktı `all_coco.json` olarak Google Drive `furniture-retrieval` klasörüne yazılır. Üretim akışı tamamen streaming yapar, hiçbir görsel kaydetmez ya da çizim üretmez.


In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
%pip install -q ultralytics==8.2.103 pandas pyarrow pillow tqdm


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path

# Dizin ayarları (gerekirse güncelle)
DRIVE_BASE = Path(DRIVE_ROOT)
IMAGES_ROOT = Path("/content/data/images")  # content/data/images/{folder}/{image_name}.png
OUTPUT_COCO = DRIVE_BASE / "all_coco.json"
TEMP_DIR = Path("/content/tmp_coco")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Model ve çalışma parametreleri
CONF_THRESHOLD = 0.15
IOU_THRESHOLD = 0.5
RUN_LIMIT = None  # Test için örn. 200 verin; tüm veri için None
NUM_WORKERS = 0   # YOLO-World predict tek thread; Colab GPU önerilir


In [ ]:
import os
import json
import torch
import numpy as np
from tqdm import tqdm
from datetime import datetime
from ultralytics import YOLOWorld


def json_safe(obj):
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.item()
    return str(obj)


def build_categories():
    hierarchy = {
        "Furniture": ["sofa", "couch", "armchair", "loveseat", "chair", "dining chair", "office chair", "stool", "bench", "ottoman", "pouf", "dining table", "coffee table", "side table", "nightstand", "bedside table", "desk", "computer desk", "console table", "wardrobe", "closet", "cabinet", "cupboard", "chest of drawers", "dresser", "bookshelf", "bookcase", "TV stand", "media unit", "bed frame", "headboard", "coat rack", "shoe rack"],
        "Textile": ["rug", "carpet", "floor mat", "runner", "doormat", "curtain", "drape", "window blind", "pillow", "cushion", "throw pillow", "blanket", "throw blanket", "duvet", "quilt", "comforter", "bed spread", "messy bed sheets", "bedding", "crumpled blanket", "towel", "bath mat"],
        "Lighting": ["floor lamp", "standing lamp", "reading lamp", "table lamp", "desk lamp", "bedside lamp", "ceiling light", "pendant light", "chandelier", "wall sconce", "wall lamp", "lampshade"],
        "Decoration": ["vase", "flower pot", "planter", "potted plant", "indoor plant", "mirror", "wall mirror", "standing mirror", "picture frame", "photo frame", "painting", "wall art", "poster", "sculpture", "statue", "figurine", "bust", "candle", "candlestick", "candle holder", "decorative bowl", "tray", "basket", "storage box", "wall clock", "table clock"],
        "Appliance": ["television", "tv screen", "monitor", "display", "refrigerator", "fridge", "washing machine", "dryer", "dishwasher", "oven", "stove", "microwave", "toaster", "kettle", "coffee maker", "speaker", "soundbar", "air conditioner", "fan", "heater", "radiator"],
    }
    cats = []
    idx = 1
    for supercat, items in hierarchy.items():
        for item in items:
            cats.append({"id": idx, "name": item, "supercategory": supercat})  # name: leaf, supercategory: parent
            idx += 1
    return cats


CATEGORIES = build_categories()
CATEGORY_NAMES = [c["name"] for c in CATEGORIES]



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = YOLOWorld("yolov8l-worldv2.pt")
model.set_classes(CATEGORY_NAMES)
model.to(device)
model.fuse()  # küçük hız kazanımı


In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}

def iter_image_paths(root: Path, limit: int | None = None):
    count = 0
    for folder in sorted(os.listdir(root)):
        folder_path = root / folder
        if not folder_path.is_dir():
            continue
        for img_path in sorted(folder_path.iterdir()):
            if not img_path.is_file() or img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            count += 1
            yield count, folder, img_path
            if limit and count >= limit:
                return


In [ ]:
def yolo_infer_and_stream(
    model: YOLOWorld,
    images_root: Path,
    output_path: Path,
    temp_dir: Path,
    limit: int | None = None,
    conf: float = 0.15,
    iou: float = 0.5,
):
    images_jsonl = temp_dir / "images.jsonl"
    annotations_jsonl = temp_dir / "annotations.jsonl"

    # Temizlik
    for fp in [images_jsonl, annotations_jsonl, output_path]:
        if fp.exists():
            fp.unlink()

    ann_id = 0
    with open(images_jsonl, "w") as img_f, open(annotations_jsonl, "w") as ann_f:
        for img_id, folder, img_path in tqdm(iter_image_paths(images_root, limit), desc="YOLO-World", unit="img"):
            results = model.predict(
                str(img_path),
                conf=conf,
                iou=iou,
                verbose=False,
                device=device,
            )
            result = results[0]
            h, w = result.orig_shape

            rel_name = str(Path(folder) / img_path.name)
            img_rec = {
                "id": img_id,
                "file_name": rel_name,
                "width": int(w),
                "height": int(h),
            }
            img_f.write(json.dumps(img_rec, default=json_safe) + "\n")

            if len(result.boxes):
                for box in result.boxes:
                    cls_idx = int(box.cls)
                    x1, y1, x2, y2 = map(float, box.xyxy[0].tolist())
                    bw, bh = x2 - x1, y2 - y1
                    ann_id += 1
                    ann = {
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": CATEGORIES[cls_idx]["id"],
                        "bbox": [x1, y1, bw, bh],
                        "area": bw * bh,
                        "segmentation": [[x1, y1, x2, y1, x2, y2, x1, y2]],
                        "score": float(box.conf),
                        "iscrowd": 0,
                    }
                    ann_f.write(json.dumps(ann, default=json_safe) + "\n")

    info = {
        "description": "YOLO-World COCO export",
        "version": "1.0",
        "year": datetime.utcnow().year,
        "date_created": datetime.utcnow().isoformat(),
    }

    # JSON streaming birleşimi
    with open(output_path, "w") as out_f:
        out_f.write("{")
        out_f.write("\"info\": " + json.dumps(info) + ", ")
        out_f.write("\"licenses\": [], ")
        out_f.write("\"categories\": " + json.dumps(CATEGORIES) + ", ")

        out_f.write("\"images\": [")
        first = True
        with open(images_jsonl, "r") as img_f:
            for line in img_f:
                if not first:
                    out_f.write(",")
                out_f.write(line.strip())
                first = False
        out_f.write("], ")

        out_f.write("\"annotations\": [")
        first = True
        with open(annotations_jsonl, "r") as ann_f:
            for line in ann_f:
                if not first:
                    out_f.write(",")
                out_f.write(line.strip())
                first = False
        out_f.write("]}")

    print(f"COCO yazıldı: {output_path}")


In [ ]:
yolo_infer_and_stream(
    model=model,
    images_root=IMAGES_ROOT,
    output_path=OUTPUT_COCO,
    temp_dir=TEMP_DIR,
    limit=RUN_LIMIT,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
)

print("Bitti ->", OUTPUT_COCO)
